In [1]:
%load_ext autoreload
%autoreload 2

### Import and run model

In [1]:
# Forward pass with an untrained model.
from cs336_basics.model import TransformerLM
import torch

# device=torch.device("cuda")
device = torch.device("mps")
model = TransformerLM(
  vocab_size=10000,
  context_length=256,
  num_layers=8,
  d_model=768,
  num_heads=16,
  d_ff=1344,
  rope_theta=10000.0,
  device=device,
  dtype=torch.float32,
)

input = torch.randint(0, 10000, (1, 256), device=device)
output = model(input)
print(output.shape)

torch.Size([1, 256, 10000])


In [ ]:
# Generate text using a trained model.
from cs336_basics.generation import generate_text
from cs336_basics.tokenizer import BPETokenizer
from cs336_basics.model import TransformerLM
from cs336_basics.io import ROOT_PATH, load_checkpoint

tok = BPETokenizer.load(ROOT_PATH / "tokenizer/tinystories_train_10000.pt")

model, _, _ = load_checkpoint(
  ROOT_PATH / "model/TinyStories/volcanic-dream-4/checkpoint_39999.pt",
  model_class=TransformerLM,
)
text = generate_text(
  model=model,
  tokenizer=tok,
  input_text= "Costanza",
  max_new_tokens=100,
  temperature=1.0,
  top_p=0.0,
)
print(model._init_args)
text

### Models to benchmark

In [ ]:
from benchmark import MODELS, instantiate_model, model_size_mb

for name, model_config in MODELS.items():
  model = instantiate_model(model_config, 256)
  total_size_mb = model_size_mb(model)
  print(f"{name} model size: {total_size_mb:.2f} MB")

# small model size: 491.42 MB
# medium model size: 1615.82 MB
# large model size: 3700.26 MB
# xl model size: 7625.66 MB
# 2.7B model size: 12998.45 MB

### Basic Benchmark: Timeit

In [ ]:
%%bash
cd ..
uv run modal run cs336_systems/benchmark_script_modal.py \
    --model-name="large" \
    --warmup-steps=5 \
    --no-synchronize \
    --no-measure-also-backward

uv run modal run cs336_systems/benchmark_script_modal.py \
    --model-name="large" \
    --warmup-steps=5 \
    --synchronize \
    --no-measure-also-backward

uv run modal run cs336_systems/benchmark_script_modal.py \
    --model-name="large" \
    --warmup-steps=5 \
    --synchronize \
    --measure-also-backward

In [ ]:
# Here are all the results for all the models.

# warmup_steps = 0. Notice the high std.
# --- small model (491.42 MB) ---
# without_sync_without_backward: 0.0892s ± 0.1655s
# with_sync_without_backward: 0.0303s ± 0.0024s
# with_sync_with_backward: 0.1095s ± 0.1056s
# --- medium model (1615.82 MB) ---
# without_sync_without_backward: 0.0818s ± 0.0434s
# with_sync_without_backward: 0.0659s ± 0.0001s
# with_sync_with_backward: 0.1975s ± 0.0004s
# --- large model (3700.26 MB) ---
# without_sync_without_backward: 0.1123s ± 0.0449s
# with_sync_without_backward: 0.1399s ± 0.0002s
# with_sync_with_backward: 0.4234s ± 0.0001s
# --- xl model (7625.66 MB) ---
# without_sync_without_backward: 0.2115s ± 0.1135s
# with_sync_without_backward: 0.2845s ± 0.0001s
# with_sync_with_backward: 0.8637s ± 0.0622s
# --- 2.7B model (12998.45 MB) ---
# without_sync_without_backward: 0.1527s ± 0.0012s
# with_sync_without_backward: 0.4181s ± 0.0001s
# with_sync_with_backward: 1.2662s ± 0.0234s

# warmup_steps = 5. Much lower std than before.
# --- small model (491.42 MB) ---
# without_sync_without_backward: 0.0312s ± 0.0002s
# with_sync_without_backward: 0.0340s ± 0.0032s
# with_sync_with_backward: 0.0767s ± 0.0008s
# --- medium model (1615.82 MB) ---
# without_sync_without_backward: 0.0633s ± 0.0013s
# with_sync_without_backward: 0.0670s ± 0.0014s
# with_sync_with_backward: 0.1997s ± 0.0014s
# --- large model (3700.26 MB) ---
# without_sync_without_backward: 0.0975s ± 0.0019s
# with_sync_without_backward: 0.1409s ± 0.0001s
# with_sync_with_backward: 0.4241s ± 0.0001s
# --- xl model (7625.66 MB) ---
# without_sync_without_backward: 0.1711s ± 0.0007s
# with_sync_without_backward: 0.2852s ± 0.0001s
# with_sync_with_backward: 0.8440s ± 0.0004s
# --- 2.7B model (12998.45 MB) ---
# without_sync_without_backward: 0.1548s ± 0.0024s
# with_sync_without_backward: 0.4184s ± 0.0001s
# with_sync_with_backward: 1.2589s ± 0.0012s

### PyTorch Profiler

In [12]:
%%bash
# Generate the profile traces (in a way that is comparable to the timeit
# approach, i.e. with synchronization and without measuring the optimizer step).
cd ..
export TERM=dumb # Tell Modal this is not an interactive terminal.
uv run modal run scripts/run_profiling.py \
  --model-name="small" \
  --warmup-steps=3 \
  --active-steps=10 \
  --synchronize \
  --no-measure-optimizer

uv run modal run scripts/run_profiling.py \
  --model-name="medium" \
  --warmup-steps=3 \
  --active-steps=10 \
  --synchronize \
  --no-measure-optimizer

uv run modal run scripts/run_profiling.py \
  --model-name="large" \
  --warmup-steps=3 \
  --active-steps=10 \
  --synchronize \
  --no-measure-optimizer

uv run modal run scripts/run_profiling.py \
  --model-name="xl" \
  --warmup-steps=3 \
  --active-steps=10 \
  --synchronize \
  --no-measure-optimizer

uv run modal run scripts/run_profiling.py \
  --model-name="2.7B" \
  --warmup-steps=3 \
  --active-steps=10 \
  --synchronize \
  --no-measure-optimizer

✓ Initialized. View run at 
https://modal.com/apps/niccsacchi/main/ap-v5M0q75KQjSmerW5066rQU
✓ Created objects.
├── 🔨 Created mount 
│   /Users/niccolosacchi/assignment2-systems/scripts/run_profiling.py
├── 🔨 Created mount PythonPackage:cs336_systems
└── 🔨 Created function run_func.
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                     Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
    cudaDeviceSynchronize       100.00%       8.740us       100.00%       8.740us       8.740us             1  
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
Self CPU time total: 8.740us

Trace saved to /traces/small/20260422_205254/modal_2.1776891179763586779.pt.trace.json
Stopping app - local entrypoint complet

[modal-client] 2026-04-22T22:53:47+0200 Timed out waiting for final app logs.


✓ App completed. View run at 
https://modal.com/apps/niccsacchi/main/ap-IdlCYlkcB0ttze0FlxfmMP
✓ Initialized. View run at 
https://modal.com/apps/niccsacchi/main/ap-9l4m1SDqEVymg16w18F7G0
✓ Created objects.
├── 🔨 Created mount 
│   /Users/niccolosacchi/assignment2-systems/scripts/run_profiling.py
├── 🔨 Created mount PythonPackage:cs336_systems
└── 🔨 Created function run_func.
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                     Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
    cudaDeviceSynchronize       100.00%       8.728us       100.00%       8.728us       8.728us             1  
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
Self CPU time total: 8.728us

Trace saved to /traces/large/20

[modal-client] 2026-04-22T22:55:46+0200 Timed out waiting for final app logs.


✓ App completed. View run at 
https://modal.com/apps/niccsacchi/main/ap-ReqjfW4I8WoEmsYV0c3HQF
✓ Initialized. View run at 
https://modal.com/apps/niccsacchi/main/ap-NFm99L3M4pdVCuegfW4PBQ
✓ Created objects.
├── 🔨 Created mount 
│   /Users/niccolosacchi/assignment2-systems/scripts/run_profiling.py
├── 🔨 Created mount PythonPackage:cs336_systems
└── 🔨 Created function run_func.
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                     Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
    cudaDeviceSynchronize       100.00%      25.854us       100.00%      25.854us      25.854us             1  
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
Self CPU time total: 25.854us

Trace saved to /traces/2.7B/20

In [15]:
%%bash
# Download the profile traces from Modal to the local machine.
export TERM=dumb # Tell Modal this is not an interactive terminal.

uv run modal volume get cs336-systems-volume \
  small/20260422_205254/modal_2.1776891179763586779.pt.trace.json \
  ~/Downloads/traces/small.trace.json --force

uv run modal volume get cs336-systems-volume \
  medium/20260422_205322/modal_2.1776891211327914121.pt.trace.json \
  ~/Downloads/traces/medium.trace.json --force

uv run modal volume get cs336-systems-volume \
  large/20260422_205401/modal_2.1776891254347928161.pt.trace.json \
  ~/Downloads/traces/large.trace.json --force

uv run modal volume get cs336-systems-volume \
  xl/20260422_205448/modal_2.1776891308911727031.pt.trace.json\
  ~/Downloads/traces/xl.trace.json --force

uv run modal volume get cs336-systems-volume \
  2.7B/20260422_205600/modal_2.1776891393470086178.pt.trace.json \
  ~/Downloads/traces/2.7B.trace.json --force

✓ Finished downloading files to local!
✓ Finished downloading files to local!
✓ Finished downloading files to local!
✓ Finished downloading files to local!
✓ Finished downloading files to local!


### Timeit vs Profiling Summary

In [ ]:
import pandas as pd

cols = ["model_name", "measurement", "time"]
df = pd.DataFrame(columns=cols)

# Timeit results. 
df.loc[len(df)] = ["small", "timeit", "0.1095s ± 0.1056s"]
df.loc[len(df)] = ["medium", "timeit", "0.1997s ± 0.0014s"]
df.loc[len(df)] = ["large", "timeit", "0.4241s ± 0.0001s"]
df.loc[len(df)] = ["xl", "timeit", "0.8440s ± 0.0004s"]
df.loc[len(df)] = ["2.7B", "timeit", "1.2589s ± 0.0012s"]

# Profiler results. Without std (quite tedious to compute from the traces).
# Running the profiler introduces a lot of overhead.
df.loc[len(df)] = ["small", "profiler", "0.2794s"]
df.loc[len(df)] = ["medium", "profiler", "0.6005s"]
df.loc[len(df)] = ["large", "profiler", "1.058s"]
df.loc[len(df)] = ["xl", "profiler", "2.028s"]
df.loc[len(df)] = ["2.7B", "profiler", "2.827s"]

df

,model_name,measurement,time
0,small,timeit,0.1095s ± 0.1056s
1,medium,timeit,0.1997s ± 0.0014s
2,large,timeit,0.4241s ± 0.0001s
3,xl,timeit,0.8440s ± 0.0004s
4,2.7B,timeit,1.2589s ± 0.0012s
5,small,profiler,0.2794s
6,medium,profiler,0.6005s
7,large,profiler,1.058s
8,xl,profiler,2.028s
9,2.7B,profiler,2.827s
